# How to fit a spatial logit model

Binary outcomes with spatial dependence. `SARLogit` is the reduced-form SAR: the
spatial multiplier acts on the mean of the log-odds. It samples by Pólya–Gamma
Gibbs and **has no NUTS path** — `fit(sampler="nuts")` raises. The equation and
constructor arguments are in [Supported Models](../models.md).

There is no spatial logit with random effects and no SEM logit. In a
cross-section the effects' sd trades off against $\beta$ and against the spatial
parameter, and binary outcomes barely constrain the trade, so their posteriors
are mostly prior.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd

from neighbayes.dgp import simulate_sar_logit
from neighbayes.models import SARLogit

## Simulate a known process

`simulate_sar_logit` builds a **square grid of side `n`**, so `n=35` gives
$35^2 = 1225$ observations. That matters: binary outcomes carry far less
information per observation than continuous ones, and $\rho$ in a spatial logit
is weakly identified below roughly a thousand units. Fitting one of these to
$n = 100$ and concluding the sampler is broken is a common mistake.

In [ ]:
RHO_TRUE, BETA_TRUE = 0.5, np.array([-0.3, 0.5])

data_sar = simulate_sar_logit(n_side=35, rho=RHO_TRUE, beta=BETA_TRUE, seed=42)
y_sar, X_sar, W_sar = data_sar["y"], data_sar["X"], data_sar["W_graph"]

print(f"observations : {len(y_sar)}  (35 x 35 queen grid)")
print(f"covariates   : {X_sar.shape[1]}")
print(f"outcome      : {int(y_sar.sum())} ones, {int(len(y_sar) - y_sar.sum())} zeros")

## Fit the reduced-form model

The sweep has three blocks, only one of which is non-conjugate:

1. $\omega \mid \rho, \beta$ — Pólya–Gamma augmentation at
   $\eta = (I - \rho W)^{-1}X\beta$
2. $\rho \mid \omega$ — a **collapsed** one-dimensional slice, with $\beta$
   integrated out
3. $\beta \mid \omega, \rho$ — conjugate normal

Collapsing block 2 is what makes this mix: $\rho$ and $\beta$ trade off in the
mean, and a $\rho$ update conditioned on the current $\beta$ crawls.

In [ ]:
model_sar = SARLogit(y=y_sar, X=X_sar, W=W_sar)
idata_sar = model_sar.fit(
    draws=1500, tune=800, chains=4, random_seed=42, progressbar=False
)

az.summary(idata_sar, var_names=["rho", "beta"]).round(3)

In [ ]:
print(f"rho  : {float(idata_sar.posterior['rho'].mean()):.3f}   (true {RHO_TRUE})")
print(
    f"beta : {idata_sar.posterior['beta'].mean(dim=['chain', 'draw']).values.round(3)}"
    f"   (true {BETA_TRUE})"
)

$\rho$ comes back with a wide credible interval even at $n = 1225$. That is
the model, not the sampler — check `ess_bulk` and `r_hat` above to confirm the
chain is healthy, then read the width as what the data support.

## Read the spillovers

In a linear SAR model $\beta$ is a marginal effect. **Here it is not.** The
link is nonlinear, so a change in $x_i$ moves $y_j$ through both the spatial
multiplier $(I - \rho W)^{-1}$ and the logistic derivative, which varies with
$\eta_i$. `impacts()` does the averaging and returns the
LeSage–Pace decomposition — direct, indirect, total — on the probability scale.

In [ ]:
model_sar.impacts().round(4)

Report these, not the raw coefficients, whenever the question is
"how much does $x$ change the outcome". The gap between `direct` and the
coefficient is the part of the effect that the spatial multiplier feeds back
onto the unit itself.

## Configure the sampler

`backend` behaves as it does everywhere in the package: `"auto"`
(the default — JAX when installed, else NumPy), `"jax"`, or `"numpy"`.

Beyond the shared `fit()` arguments, `SARLogit` accepts the options below.
Anything else raises `TypeError` rather than being silently ignored.

| Option | Meaning |
|---|---|
| `slice_width` | initial slice interval for ρ |
| `init_jitter` | scale of the random perturbation applied to chain starts |
| `krylov_degree`, `krylov_dmax`, `krylov_reuse` | shift-invert Krylov basis for the ρ conditional — degree, trust radius, and whether the basis is reused across draws |
| `timeout` | abort a run that exceeds this many seconds |

In [ ]:
idata_tuned = SARLogit(y=y_sar, X=X_sar, W=W_sar).fit(
    draws=800,
    tune=500,
    chains=2,
    random_seed=7,
    progressbar=False,
    backend="numpy",
    slice_width=0.4,
    krylov_reuse=True,
)
print(
    f"rho = {float(idata_tuned.posterior['rho'].mean()):.3f}, "
    f"ess = {float(az.ess(idata_tuned, var_names=['rho'])['rho']):.0f}"
)

## Fitted probabilities

`fitted_probabilities()` returns $P(y_i = 1)$ at the posterior mean,
$\hat\eta = (I - \hat\rho W)^{-1} X\hat\beta$: the multiplier is in the mean.

In [ ]:
probs_sar = model_sar.fitted_probabilities()

pd.Series(
    {"mean": probs_sar.mean(), "min": probs_sar.min(), "max": probs_sar.max()},
    name="SARLogit",
).round(3)

## Fit the latent SAR and SEM

The reduced form lets the multiplier carry only the covariates. The **latent
SAR** carries the logit's own disturbance too, $y = \mathbf 1\{y^* > 0\}$ with
$(I - \rho W)y^* = X\beta + \varepsilon$ and $\varepsilon$ standard logistic, so
neighbours share their unobservables as well as their covariates; `SARLogit`
fits it with `latent=True`. Its error-only sibling, $y^* = X\beta +
(I - \lambda W)^{-1}\varepsilon$, is `SEMLogit`. At $\rho = 0$ or $\lambda = 0$
both are the plain logit, and neither has a variance parameter to trade off
against $\rho$.

Both sample by an exact Gibbs sweep over the latent utilities and the
Kolmogorov scales that make the logistic a normal scale mixture (this needs
`augax`). They have no NUTS path and store no pointwise log-likelihood.

In [ ]:
from neighbayes.dgp import simulate_sem_logit
from neighbayes.models import SEMLogit

data_lat = simulate_sar_logit(n_side=30, rho=0.4, beta=BETA_TRUE, seed=3, latent=True)
latent = SARLogit(
    y=data_lat["y"], X=data_lat["X"], W=data_lat["W_graph"], latent=True
)
idata_lat = latent.fit(draws=1000, tune=500, chains=2, random_seed=3, progressbar=False)
az.summary(idata_lat, var_names=["rho", "beta"]).round(3)

In [ ]:
data_sem = simulate_sem_logit(n_side=30, lam=0.4, beta=BETA_TRUE, seed=4)
sem = SEMLogit(y=data_sem["y"], X=data_sem["X"], W=data_sem["W_graph"])
idata_sem = sem.fit(draws=1000, tune=500, chains=2, random_seed=4, progressbar=False)
az.summary(idata_sem, var_names=["lam", "beta"]).round(3)

The latent models' impacts are on the latent index (`scale="probability"`
raises for them), and `fitted_probabilities()` simulates the utilities: the
multiplier mixes the disturbances, so their marginals are no longer logistic.
The GLM suite's LM tests on a `Logit` fit are the score tests of exactly these
two models, LM-Lag of the latent SAR and LM-Error of the latent SEM; see
[Supported Models](../models.md).

## What to check before trusting the output

In [ ]:
print("groups:", list(idata_sar.children))
az.summary(idata_sar, var_names=["rho"]).round(3)

:::{caution}
`az.loo` and `az.waic` run on this output and should not be used to choose
between a lag and an error specification. Pointwise leave-one-out assumes
observations are exchangeable given the parameters; under a spatial lag they are
not, because removing $y_i$ changes the implied mean of its neighbours through
$(I - \rho W)^{-1}$. Use the
[Bayesian LM specification tests](bayesian_lmtests.ipynb) or
[spatial block cross-validation](spatial_cv_demo.ipynb) instead.
:::

## See also

- [How to run Bayesian LM specification tests](bayesian_lmtests.ipynb) —
  deciding between lag and error before you fit
- [How to set priors](priors.ipynb) — `SARLogitPriors`
- [How to estimate spatial negative binomial models](negbin_estimation.ipynb) —
  the same Pólya–Gamma machinery for counts